In [3]:
# Ensure you have tensorflow, gymnasium, pettingzoo, and numpy installed:
# pip install tensorflow gymnasium pettingzoo numpy
# For visual rendering: pip install pygame

import gymnasium as gym
import pettingzoo.classic.chess_v6 as chess_env
import numpy as np
import random
import collections
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

# Set a random seed for reproducibility (optional)
# tf.random.set_seed(42)
# np.random.seed(42)
# random.seed(42)

# --- 1. Deep Q-Network (DQN) Agent ---
class DQNAgent:
    """
    A simplified Deep Q-Network agent for the PettingZoo Chess environment.
    This agent uses a neural network to approximate Q-values.
    """
    def __init__(self, observation_shape, action_size, learning_rate=0.001,
                 gamma=0.99, epsilon=1.0, epsilon_decay=0.995, epsilon_min=0.01,
                 replay_buffer_size=10000):
        
        self.observation_shape = observation_shape # (8, 8, 111) for board state
        self.action_size = action_size             # 4672 possible chess moves
        self.lr = learning_rate
        self.gamma = gamma                         # Discount factor
        self.epsilon = epsilon                     # Exploration rate
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min

        # Replay buffer for experience replay (stores (s, a, r, s', done) tuples)
        self.replay_buffer = collections.deque(maxlen=replay_buffer_size)
        
        # Build the Q-Network
        self.q_network = self._build_q_network()
        self.optimizer = optimizers.Adam(learning_rate=self.lr)
        self.loss_fn = tf.keras.losses.MeanSquaredError()

        print(f"DQN Agent initialized:")
        print(f"  Observation Shape: {self.observation_shape}, Action Size: {self.action_size}")
        print(f"  Hyperparameters: LR={self.lr}, Gamma={self.gamma}, Epsilon={self.epsilon} (decay={self.epsilon_decay}, min={self.epsilon_min})")
        print(f"  Replay Buffer Size: {replay_buffer_size}")
        print("  Q-Network Summary:")
        self.q_network.summary()

    def _build_q_network(self):
        """
        Builds a simple Convolutional Neural Network (CNN) for Q-value approximation.
        The architecture is kept relatively shallow for demonstration purposes.
        """
        model = models.Sequential([
            # Input layer: (8, 8, 111) as defined by PettingZoo Chess board observation
            layers.Input(shape=self.observation_shape),
            
            # Convolutional layers to process the board state (spatial features)
            layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu', padding='same'),
            layers.BatchNormalization(),
            layers.Conv2D(filters=64, kernel_size=(3, 3), activation='relu', padding='same'),
            layers.BatchNormalization(),
            layers.Conv2D(filters=64, kernel_size=(3, 3), activation='relu', padding='same'),
            layers.BatchNormalization(),
            
            layers.Flatten(), # Flatten the output of the convolutional layers into a 1D vector

            # Dense layers for higher-level feature processing
            layers.Dense(512, activation='relu'),
            layers.Dropout(0.2), # Dropout for regularization to prevent overfitting
            layers.Dense(256, activation='relu'),
            
            # Output layer: Q-values for each possible action. Linear activation for Q-values.
            layers.Dense(self.action_size, activation='linear') 
        ], name="Q_Network")
        
        return model

    def store_experience(self, state, action, reward, next_state, terminated):
        """Stores an experience tuple in the replay buffer."""
        # 'state' and 'next_state' here are the full observation dictionaries from pettingzoo.
        # We store them as-is and extract the 'observation' (board state) part during learning.
        self.replay_buffer.append((state, action, reward, next_state, terminated))

    def choose_action(self, observation_dict, legal_actions_mask):
        """
        Selects an action using an epsilon-greedy policy, respecting legal moves.
        
        Args:
            observation_dict (dict): The full observation dictionary from env.last(),
                                     containing 'observation' (board state) and 'action_mask'.
            legal_actions_mask (np.array): A boolean array (size 4672) where True indicates a legal move.
        
        Returns:
            int: The chosen action.
        """
        # Get only the legal actions from the mask
        legal_indices = np.where(legal_actions_mask == 1)[0]
        
        if len(legal_indices) == 0:
            # This case means there are no legal moves available, which should imply game termination
            # (e.g., stalemate or checkmate already handled by the environment).
            # If we reach here, it's a safeguard against situations where `terminated` wasn't immediately true
            # but no moves are possible.
            print("Warning: choose_action called with no legal actions. This usually means the game is over.")
            return None 

        if random.random() < self.epsilon:
            # Explore: Choose a random legal action from the available ones
            action = random.choice(legal_indices)
        else:
            # Exploit: Choose the best action based on Q-values predicted by the network
            # Extract the actual board observation part for the network input
            board_observation = observation_dict['observation']
            # Add a batch dimension (e.g., (8,8,111) -> (1,8,8,111)) for single prediction
            obs_tensor = tf.convert_to_tensor(board_observation[np.newaxis, ...], dtype=tf.float32)
            q_values = self.q_network(obs_tensor).numpy().flatten()
            
            # Mask out illegal actions by setting their Q-values to a very small negative number.
            # This ensures np.argmax will never select them.
            q_values[legal_actions_mask == 0] = -1e10 

            action = np.argmax(q_values)
            
            # Sanity check: Ensure the chosen action is actually legal
            if not legal_actions_mask[action]:
                print(f"CRITICAL ERROR: Chosen action {action} was illegal despite masking! Falling back to random legal.")
                action = random.choice(legal_indices) # Fallback to a random legal action

        return action

    def learn(self, batch_size=32):
        """
        Trains the Q-network using a batch of experiences sampled from the replay buffer.
        """
        if len(self.replay_buffer) < batch_size:
            # Not enough experiences in the buffer to sample a full batch for training
            return

        # Sample a random batch of experiences (s, a, r, s', done)
        batch = random.sample(self.replay_buffer, batch_size)
        
        # Unpack the batch and prepare tensors for the neural network
        # For states and next_states, extract the 'observation' part (the board state)
        states_obs = np.array([exp[0]['observation'] for exp in batch])
        actions = np.array([exp[1] for exp in batch])
        rewards = np.array([exp[2] for exp in batch])
        next_states_obs = np.array([exp[3]['observation'] for exp in batch])
        terminateds = np.array([exp[4] for exp in batch])

        # Convert NumPy arrays to TensorFlow tensors for GPU processing
        states_tensor = tf.convert_to_tensor(states_obs, dtype=tf.float32)
        actions_tensor = tf.convert_to_tensor(actions, dtype=tf.int32)
        rewards_tensor = tf.convert_to_tensor(rewards, dtype=tf.float32)
        next_states_tensor = tf.convert_to_tensor(next_states_obs, dtype=tf.float32)
        terminateds_tensor = tf.convert_to_tensor(terminateds, dtype=tf.float32) # Boolean to float (0.0 or 1.0)

        # Calculate target Q-values (Bellman Equation: R + gamma * max_Q(S', A'))
        # Get Q-values for the next states from the current Q-network
        next_q_values = self.q_network(next_states_tensor)
        # Find the maximum Q-value for each next state
        max_next_q = tf.reduce_max(next_q_values, axis=1) 

        # Compute the target Q-values:
        # If the episode terminated (done=True), the future reward (gamma * max_Q) is zero.
        target_q_values = rewards_tensor + self.gamma * max_next_q * (1 - terminateds_tensor)

        # Use GradientTape to record operations for automatic differentiation
        with tf.GradientTape() as tape:
            # Get the predicted Q-values from the network for the *states* in the batch
            current_q_values_full = self.q_network(states_tensor)
            # Select only the Q-values corresponding to the *actions actually taken* in the batch
            action_indices = tf.stack([tf.range(batch_size), actions_tensor], axis=1)
            current_q_values = tf.gather_nd(current_q_values_full, action_indices)

            # Compute the Mean Squared Error (MSE) loss
            loss = self.loss_fn(target_q_values, current_q_values)
        
        # Compute gradients of the loss with respect to the Q-network's trainable variables
        gradients = tape.gradient(loss, self.q_network.trainable_variables)
        # Apply the gradients to update the network's weights
        self.optimizer.apply_gradients(zip(gradients, self.q_network.trainable_variables))
        
    def decay_epsilon(self):
        """Decays epsilon over time to reduce exploration."""
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)


# --- 2. Training Function ---
def train_drl_chess(num_episodes=500):
    """
    Trains a simplified DQN agent to play chess against a random opponent.
    """
    print(f"\n--- Starting DRL Chess Training (Episodes: {num_episodes}) ---")

    # Initialize PettingZoo Chess Environment
    # render_mode="ansi" for text-based board in console (default if not specified)
    # render_mode="human" for visual window (requires pygame)
    env = chess_env.env(render_mode="ansi")
    
    # Get environment specs for agent initialization
    env.reset() # Must call reset once to populate observation/action spaces
    
    # Correctly access the 'observation' part of the observation space for the network input shape
    observation_shape = env.observation_space('player_0')['observation'].shape
    action_size = env.action_space('player_0').n

    print(f"DEBUG: Retrieved observation_shape for network input: {observation_shape}")
    print(f"DEBUG: Retrieved action_size: {action_size}")
    
    env.close() # Close after getting specs

    # Create our DQN agent (e.g., for White player)
    white_agent = DQNAgent(observation_shape, action_size)

    # Training parameters
    BATCH_SIZE = 64
    TRAIN_START_SIZE = 500 # Start training only after collecting some experiences in the replay buffer

    # Dictionary to accumulate rewards per agent per episode
    rewards_per_episode = collections.defaultdict(float) 

    for episode in range(num_episodes):
        print(f"\n===== Episode {episode + 1}/{num_episodes} =====")
        env.reset(seed=random.randint(0, 100000)) # Reset for a new game with a random seed
        
        # Loop through turns within an episode using env.agent_iter()
        # This will yield the agent_id whose turn it is.
        for agent_id in env.agent_iter():
            # Get current observation, reward, termination status, and info for this agent's turn
            observation_dict, reward, terminated, truncated, info = env.last()
            
            # --- Handle game termination first ---
            if terminated or truncated:
                # The game is over for this episode (e.g., checkmate, stalemate, 50-move rule).
                # The reward received here (`reward`) is the final reward for the agent who just finished their turn.
                # Update final rewards for all agents from `env.rewards` property.
                for final_agent_id in env.agents:
                    rewards_per_episode[final_agent_id] = env.rewards[final_agent_id]
                break # Exit the `agent_iter` loop, as the episode has concluded.
            
            # --- If the game is NOT terminated, proceed with action selection ---
            # At this point, `info["action_mask"]` *should* be available.
            legal_actions_mask = info["action_mask"]
            
            # Ensure there are legal actions. If not, it's an unexpected termination (e.g., stalemate not caught earlier).
            if np.sum(legal_actions_mask) == 0:
                print(f"Warning: Agent {agent_id} has no legal actions available, forcing episode termination.")
                # Force termination and break, distributing current rewards.
                for final_agent_id in env.agents:
                    rewards_per_episode[final_agent_id] = env.rewards[final_agent_id]
                break # Exit the `agent_iter` loop

            # --- Choose action based on agent_id ---
            action = None # Initialize action
            if agent_id == 'player_0': # Our DQN agent (White)
                action = white_agent.choose_action(observation_dict, legal_actions_mask)
            else: # Random opponent (Black)
                # Select a random action from legal ones for the opponent
                legal_indices = np.where(legal_actions_mask == 1)[0]
                action = random.choice(legal_indices)

            # Safeguard if choose_action somehow returns None (though it shouldn't if legal_actions_mask is not empty)
            if action is None:
                print(f"Error: Agent {agent_id} failed to choose a valid action. Terminating episode.")
                for final_agent_id in env.agents:
                    rewards_per_episode[final_agent_id] = env.rewards[final_agent_id]
                break # Exit the `agent_iter` loop

            # --- Step the environment with the chosen action ---
            env.step(action)

            # --- Store experience for the DQN agent ('player_0') ---
            # An experience tuple is (current_state, action_taken, reward_received, next_state, done_status).
            # In PettingZoo, the reward for the action taken by `agent_id` is typically assigned
            # when `next_agent_id`'s turn begins (or if the game ends).
            
            # This is complex in multi-agent. The simplest reliable way for this demo:
            # Store the (s, a, r', s', done') for the agent that *just acted*.
            # 's' is `observation_dict` (before action). 'a' is `action`.
            # 'r'', 's'', 'done'' are obtained by looking at `env.last()` for the `next_agent_id`.
            
            if agent_id == 'player_0':
                # Get the state and reward from the perspective of the *next* agent (which contains
                # the reward for player_0's just-completed move).
                next_agent_id_after_step = env.next_agent
                
                # Check if the next agent is still active. If not, the game ended from player_0's move.
                if next_agent_id_after_step in env.agents:
                    next_observation_dict, next_reward, next_terminated, next_truncated, _ = env.last(next_agent_id_after_step)
                    
                    white_agent.store_experience(
                        observation_dict,       # S: observation before player_0's action
                        action,                 # A: player_0's action
                        next_reward,            # R': reward received by player_0 for this action
                        next_observation_dict,  # S': observation after player_0's action
                        next_terminated or next_truncated # Done': whether the game ended after player_0's action
                    )
                    # Accumulate reward for player_0 (this is the reward for its own action)
                    rewards_per_episode[agent_id] += next_reward

                    # Train the agent if enough experiences are collected
                    if len(white_agent.replay_buffer) > TRAIN_START_SIZE:
                        white_agent.learn(BATCH_SIZE)
                else:
                    # Case: Player_0's move directly ended the game (e.g., checkmate).
                    # The final rewards will be fully captured by `env.rewards` after the loop.
                    # No experience to store in this specific `(s,a,r',s',done')` format,
                    # as there's no `next_agent_id` observation.
                    pass 

        # Decay epsilon after each episode
        white_agent.decay_epsilon()

        # Print episode summary
        print(f"Episode {episode + 1} finished.")
        # After the game loop completes (either naturally or by break), fetch final rewards.
        # env.rewards is the most reliable way to get total accumulated reward for each agent in the episode.
        
        for agent_id_final in env.agents:
            if agent_id_final in env.rewards: # Ensure agent is in the rewards dict (might not be if game ended very early)
                rewards_per_episode[agent_id_final] = env.rewards[agent_id_final]

        for agent_id, total_reward in rewards_per_episode.items():
            print(f"  Total reward for {agent_id} in this game: {total_reward:.2f}")
        rewards_per_episode.clear() # Reset for next episode

    env.close()
    print("\n--- DRL Chess Training Completed ---")
    print("This simplified DQN demonstrates how to connect a neural network")
    print("to the PettingZoo Chess environment using experience replay.")
    print("\nTo build a truly competitive chess bot, you would need:")
    print("  - A much deeper and more specialized neural network architecture.")
    print("  - A 'target network' for stable DQN training (omitted for simplicity here).")
    # Rethink the reward accumulation for each episode. `env.rewards`
    # should be used for final accumulated reward per agent. The `rewards_per_episode`
    # dictionary should be updated from `env.rewards` at the end of each episode.
    # During the episode, `rewards_per_episode[agent_id] += next_reward` is fine for
    # tracking intermediate rewards for the agent whose experience is being stored.
